# LeetCode 1775 — Equal Sum Arrays With Minimum Number of Operations bằng Python

Link đề: https://leetcode.com/problems/equal-sum-arrays-with-minimum-number-of-operations/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def minOperations(self, nums1: List[int], nums2: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Đổi bài toán hai mảng thành bài toán **thu hẹp một hiệu số** `d = sum(lớn) − sum(nhỏ)`.
2. Tính **mức "đóng góp" tối đa** của mỗi phần tử khi bị đổi: mảng tổng nhỏ tăng tới 6 (`6 − x`), mảng tổng lớn giảm về 1 (`x − 1`).
3. Áp dụng **tham lam**: luôn dùng phần tử có đóng góp lớn nhất trước; chứng minh bằng lập luận "k phần tử lớn nhất".
4. Tối ưu từ sắp xếp `O(N log N)` sang **đếm** `O(N)` vì đóng góp chỉ nằm trong `0..5`.
5. Nhận diện ngay trường hợp **bất khả thi** (`−1`) bằng so sánh khoảng giá trị.

## 2. Đề bài & ví dụ

Cho hai mảng số nguyên `nums1`, `nums2` (có thể khác độ dài), mọi giá trị trong `[1, 6]`. Một thao tác: đổi **một** phần tử bất kỳ (ở mảng nào cũng được) thành một giá trị bất kỳ trong `[1, 6]`.

Trả về số thao tác **tối thiểu** để `sum(nums1) == sum(nums2)`, hoặc `−1` nếu không thể.

**Ví dụ 1:** `nums1 = [1,2,3,4,5,6]`, `nums2 = [1,1,2,2,2,2]` → `3`. (Tổng 21 và 10, hiệu 11: đổi `nums2[0]` thành 6 (+5), `nums1[5]` thành 1 (−5), `nums1[2]` thành 2 (−1).)

**Ví dụ 2:** `nums1 = [1,1,1,1,1,1,1]`, `nums2 = [6]` → `−1` (tổng `nums1` tối thiểu 7 > tổng `nums2` tối đa 6).

**Ví dụ 3:** `nums1 = [6,6]`, `nums2 = [1]` → `3`.

**Ràng buộc:** `1 ≤ nums1.length, nums2.length ≤ 10⁵`, `1 ≤ nums1[i], nums2[i] ≤ 6`.

In [1]:
import random
import time
from itertools import product
from typing import List

examples = [([1, 2, 3, 4, 5, 6], [1, 1, 2, 2, 2, 2], 3), ([1, 1, 1, 1, 1, 1, 1], [6], -1), ([6, 6], [1], 3)]
for a, b, exp in examples:
    print(f"nums1={a} (tổng {sum(a)}), nums2={b} (tổng {sum(b)}) -> kỳ vọng {exp}")

nums1=[1, 2, 3, 4, 5, 6] (tổng 21), nums2=[1, 1, 2, 2, 2, 2] (tổng 10) -> kỳ vọng 3
nums1=[1, 1, 1, 1, 1, 1, 1] (tổng 7), nums2=[6] (tổng 6) -> kỳ vọng -1
nums1=[6, 6] (tổng 12), nums2=[1] (tổng 1) -> kỳ vọng 3


## 3. Kiến thức nền

### 3.1 Khoảng tổng có thể đạt

Mảng độ dài `L` với giá trị trong `[1, 6]` có tổng trong `[L, 6L]`. Hai mảng có thể bằng tổng ⇔ hai khoảng `[n1, 6n1]` và `[n2, 6n2]` **giao nhau** ⇔ `6·n1 ≥ n2` và `6·n2 ≥ n1`.

In [2]:
for a, b, _ in examples:
    n1, n2 = len(a), len(b)
    print(f"[{n1}, {6*n1}] và [{n2}, {6*n2}] giao nhau? {6 * n1 >= n2 and 6 * n2 >= n1}")

[6, 36] và [6, 36] giao nhau? True
[7, 42] và [1, 6] giao nhau? False
[2, 12] và [1, 6] giao nhau? True


### 3.2 Sắp xếp giảm dần và tổng tiền tố

`sorted(arr, reverse=True)` rồi cộng dồn — số phần tử đầu tiên cần để tổng `≥ d`.

In [3]:
arr = [5, 1, 4, 3, 2]; d = 8; acc = 0
for k, v in enumerate(sorted(arr, reverse=True), 1):
    acc += v
    if acc >= d:
        print("cần", k, "phần tử lớn nhất (tổng", acc, ")"); break

cần 2 phần tử lớn nhất (tổng 9 )


### 3.3 Đếm thay cho sắp xếp

Khi giá trị chỉ nằm trong một miền nhỏ (`0..5`), đếm `cnt[v]` rồi duyệt `v` từ lớn xuống nhỏ tương đương việc sắp giảm dần — `O(N + 6)`.

In [4]:
cnt = [0] * 6
for v in [5, 1, 4, 3, 2, 5]:
    cnt[v] += 1
print(cnt, "-> giảm dần:", [v for v in range(5, -1, -1) for _ in range(cnt[v])])

[0, 1, 1, 1, 1, 2] -> giảm dần: [5, 5, 4, 3, 2, 1]


### 3.4 `divmod` và phép chia lấy trần

Khi dùng cùng một đóng góp `v` nhiều lần, số lần cần để phủ `d` là `⌈d / v⌉ = (d + v − 1) // v`.

In [5]:
print((11 + 5 - 1) // 5, divmod(11, 5))

3 (2, 1)


## 4. Ý tưởng nền tảng / trực giác

Giả sử `s1 < s2` (nếu không thì đổi vai). Cần **thu hẹp** hiệu `d = s2 − s1 > 0`.

- Đổi một phần tử `x` của **nums1** (tổng nhỏ) thành 6 ⇒ `d` giảm tối đa `6 − x`.
- Đổi một phần tử `y` của **nums2** (tổng lớn) thành 1 ⇒ `d` giảm tối đa `y − 1`.
- Mỗi phần tử chỉ cần đổi **tối đa một lần** (đổi lại lần nữa là phí).
- Có thể giảm "ít hơn mức tối đa" (đổi thành giá trị trung gian) ⇒ chỉ cần tổng đóng góp `≥ d`.

Tính tay **Ví dụ 1**: `s1 = 21`, `s2 = 10` ⇒ đổi vai: nhỏ = `nums2` (10), lớn = `nums1` (21), `d = 11`.
- Đóng góp từ `nums2 = [1,1,2,2,2,2]`: `6 − x` = `[5,5,4,4,4,4]`.
- Đóng góp từ `nums1 = [1..6]`: `x − 1` = `[0,1,2,3,4,5]`.
- Sắp giảm: `5, 5, 5, 4, 4, …` ⇒ cộng dồn `5, 10, 15 ≥ 11` ⇒ **3** thao tác.

In [6]:
a, b, _ = examples[0]
small, big = (a, b) if sum(a) < sum(b) else (b, a)
gains = sorted([6 - x for x in small] + [y - 1 for y in big], reverse=True)
print("d =", sum(big) - sum(small), "| đóng góp giảm dần:", gains)

d = 11 | đóng góp giảm dần: [5, 5, 5, 4, 4, 4, 4, 4, 3, 2, 1, 0]


## 5. Cách 1 — Brute Force / baseline: thử mọi phép gán (chỉ cho mảng rất ngắn)

**Ý tưởng:** với mọi mảng đích `(a', b')` có giá trị trong `[1, 6]` (cùng độ dài), nếu `sum(a') == sum(b')` thì số thao tác là số vị trí khác với mảng gốc; lấy min. Có `6^{n1+n2}` phép gán ⇒ chỉ dùng khi `n1 + n2 ≤ 5`.

In [7]:
def ops_bruteforce(nums1, nums2):
    n1 = len(nums1); best = None
    for vals in product(range(1, 7), repeat=n1 + len(nums2)):
        a2, b2 = vals[:n1], vals[n1:]
        if sum(a2) == sum(b2):
            changes = sum(x != y for x, y in zip(a2, nums1)) + sum(x != y for x, y in zip(b2, nums2))
            best = changes if best is None else min(best, changes)
    return -1 if best is None else best

for a, b, exp in examples[1:]:
    got = ops_bruteforce(a, b) if len(a) + len(b) <= 5 else "(quá lớn)"
    print(a, b, "->", got, "| kỳ vọng", exp)

[1, 1, 1, 1, 1, 1, 1] [6] -> (quá lớn) | kỳ vọng -1
[6, 6] [1] -> 3 | kỳ vọng 3


**Độ phức tạp:** `O(6^{n1+n2} · (n1+n2))` — chỉ dùng để kiểm chứng.

### Cách 1' — Thử `k = 0, 1, 2, …` và kiểm tra "có thể với đúng k thao tác?"

Với `k` thao tác, mức thu hẹp tối đa là tổng `k` đóng góp lớn nhất. Thử `k` tăng dần, mỗi lần **sắp lại** ⇒ `O(N² log N)`.

In [8]:
def ops_try_k(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2:
        nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    gains = [6 - x for x in nums1] + [y - 1 for y in nums2]
    for k in range(len(gains) + 1):
        if sum(sorted(gains, reverse=True)[:k]) >= d:
            return k
    return -1

print([ops_try_k(a, b) for a, b, _ in examples])

[3, -1, 3]


**Nút thắt:** sắp lại và cộng lại cho mỗi `k`; trong khi chỉ cần **một** lần sắp và cộng dồn.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với `k` thao tác, mức thu hẹp tối đa đạt được = **tổng `k` đóng góp lớn nhất** (chọn `k` phần tử khác nhau, mỗi phần tử đổi một lần). Hàm này tăng theo `k` ⇒ đáp án là `k` nhỏ nhất có tổng `≥ d`.

**Quy tắc tham lam:** sắp đóng góp giảm dần, cộng dồn; dừng ở phần tử đầu tiên làm tổng `≥ d`. Nếu cộng hết vẫn `< d` ⇒ `−1`.

**Vì sao "≥ d" là đủ (không cần đúng bằng):** nếu tổng đóng góp của `k` phần tử đã chọn vượt `d`, ta giảm bớt mức đổi của phần tử cuối (chọn giá trị trung gian) để thu hẹp **đúng** `d`.

**Tối ưu:** đóng góp ∈ `{0, …, 5}` ⇒ đếm `cnt[v]`, duyệt `v = 5, 4, …, 1` và dùng `⌈d / v⌉` để lấy nhiều phần tử cùng lúc ⇒ `O(N)`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam + sắp xếp

In [9]:
def ops_sort(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 == s2:
        return 0
    if s1 > s2:
        nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    gains = sorted([6 - x for x in nums1] + [y - 1 for y in nums2], reverse=True)
    for k, g in enumerate(gains, 1):
        d -= g
        if d <= 0:
            return k
    return -1

print([ops_sort(a, b) for a, b, _ in examples])

[3, -1, 3]


**Code nộp LeetCode (Cách 2):** đổi tên thành `minOperations`. **Độ phức tạp:** Time `O(N log N)`, Space `O(N)` với `N = n1 + n2`.

### 7.2 Cách 3 — Tham lam + đếm (counting), `O(N)`

In [10]:
def ops_count(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2:
        nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    if d == 0:
        return 0
    cnt = [0] * 6
    for x in nums1: cnt[6 - x] += 1
    for y in nums2: cnt[y - 1] += 1
    ops = 0
    for v in range(5, 0, -1):
        if cnt[v] == 0:
            continue
        need = (d + v - 1) // v                  # số phần tử đóng góp v cần để phủ d
        if need <= cnt[v]:
            return ops + need
        ops += cnt[v]; d -= cnt[v] * v
    return -1

print([ops_count(a, b) for a, b, _ in examples])

[3, -1, 3]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(N)`, Space `O(1)` (mảng 6 phần tử).

### 7.3 Cách 4 — Kiểm tra khả thi trước + hai con trỏ trên hai mảng đã sắp

Kiểm tra `6·n1 < n2 or 6·n2 < n1` ⇒ `−1` ngay. Sau đó sắp mảng nhỏ **tăng dần** (đóng góp `6 − x` giảm dần) và mảng lớn **giảm dần** (đóng góp `y − 1` giảm dần), dùng hai con trỏ chọn đóng góp lớn hơn ở mỗi bước (giống trộn hai dãy đã sắp).

In [11]:
def ops_two_pointers(nums1, nums2):
    if 6 * len(nums1) < len(nums2) or 6 * len(nums2) < len(nums1):
        return -1
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2:
        nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    A = sorted(nums1); B = sorted(nums2, reverse=True)
    i = j = ops = 0
    while d > 0:
        ga = 6 - A[i] if i < len(A) else -1
        gb = B[j] - 1 if j < len(B) else -1
        if ga >= gb:
            d -= ga; i += 1
        else:
            d -= gb; j += 1
        ops += 1
    return ops

print([ops_two_pointers(a, b) for a, b, _ in examples])

[3, -1, 3]


Vòng `while d > 0` chắc chắn dừng vì đã kiểm tra khả thi (tổng mọi đóng góp `≥ d`).

**Độ phức tạp Cách 4:** Time `O(N log N)`, Space `O(N)`.

## 8. TRACE từng bước

Trace Cách 3 trên **Ví dụ 1** (sau khi đổi vai: nhỏ = `[1,1,2,2,2,2]`, lớn = `[1,2,3,4,5,6]`, `d = 11`):

| v | cnt[v] | need = ⌈d/v⌉ | đủ? | ops | d |
|---|---|---|---|---|---|
| 5 | 3 | ⌈11/5⌉ = 3 | có | **3** | — |

Trace Ví dụ 3 (`nums1 = [6,6]`, `nums2 = [1]`: nhỏ = `[1]`, lớn = `[6,6]`, `d = 11`): `cnt[5] = 3` (một từ `6 − 1`, hai từ `6 − 1` của mảng lớn) ⇒ `need = 3` ⇒ **3**.

In [12]:
def trace_count(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2:
        nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    print(f"mảng tổng nhỏ={nums1} (tổng {s1}), mảng tổng lớn={nums2} (tổng {s2}), d={d}")
    if d == 0: return 0
    cnt = [0] * 6
    for x in nums1: cnt[6 - x] += 1
    for y in nums2: cnt[y - 1] += 1
    print("cnt[v] (số phần tử có đóng góp v):", cnt)
    ops = 0
    for v in range(5, 0, -1):
        if cnt[v] == 0: continue
        need = (d + v - 1) // v
        if need <= cnt[v]:
            print(f"v={v}: cần {need} <= có {cnt[v]} -> xong, ops = {ops + need}"); return ops + need
        ops += cnt[v]; d -= cnt[v] * v
        print(f"v={v}: dùng hết {cnt[v]} phần tử -> ops={ops}, d còn {d}")
    print("hết đóng góp mà d > 0 -> -1"); return -1

for a, b, exp in examples:
    assert trace_count(a, b) == exp; print()

mảng tổng nhỏ=[1, 1, 2, 2, 2, 2] (tổng 10), mảng tổng lớn=[1, 2, 3, 4, 5, 6] (tổng 21), d=11
cnt[v] (số phần tử có đóng góp v): [1, 1, 1, 1, 5, 3]
v=5: cần 3 <= có 3 -> xong, ops = 3

mảng tổng nhỏ=[6] (tổng 6), mảng tổng lớn=[1, 1, 1, 1, 1, 1, 1] (tổng 7), d=1
cnt[v] (số phần tử có đóng góp v): [8, 0, 0, 0, 0, 0]
hết đóng góp mà d > 0 -> -1

mảng tổng nhỏ=[1] (tổng 1), mảng tổng lớn=[6, 6] (tổng 12), d=11
cnt[v] (số phần tử có đóng góp v): [0, 0, 0, 0, 0, 3]
v=5: cần 3 <= có 3 -> xong, ops = 3



## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — tham lam + đếm, `O(N)`.

In [13]:
class Solution:
    def minOperations(self, nums1: List[int], nums2: List[int]) -> int:
        s1, s2 = sum(nums1), sum(nums2)
        if s1 > s2:
            nums1, nums2, s1, s2 = nums2, nums1, s2, s1
        d = s2 - s1
        if d == 0:
            return 0
        cnt = [0] * 6
        for x in nums1:
            cnt[6 - x] += 1
        for y in nums2:
            cnt[y - 1] += 1
        ops = 0
        for v in range(5, 0, -1):
            if cnt[v] == 0:
                continue
            need = (d + v - 1) // v
            if need <= cnt[v]:
                return ops + need
            ops += cnt[v]
            d -= cnt[v] * v
        return -1

for a, b, exp in examples:
    got = Solution().minOperations(a, b); print(got, exp); assert got == exp

3 3
-1 -1
3 3


## 10. Vì sao thuật toán đúng

Giả sử `s1 < s2`, `d = s2 − s1`. Gọi `g_e` là đóng góp tối đa của phần tử `e` (`6 − x` nếu thuộc mảng nhỏ, `y − 1` nếu thuộc mảng lớn).

**Bổ đề 1 (cần).** Nếu đổi tập phần tử `E` (mỗi phần tử một lần) mà hai tổng bằng nhau thì `Σ_{e∈E} g_e ≥ d`. *Lý do:* đổi `e` làm hiệu `s2 − s1` giảm **tối đa** `g_e` (tăng phần tử mảng nhỏ tối đa tới 6, giảm phần tử mảng lớn tối đa về 1); muốn hiệu từ `d` về 0 thì tổng mức giảm phải `≥ d`. (Đổi theo hướng ngược lại chỉ làm hiệu tăng — vô ích.)

**Bổ đề 2 (đủ).** Nếu `Σ_{e∈E} g_e ≥ d` thì đổi được đúng các phần tử trong `E` để hiệu = 0: dùng hết mức cho các phần tử đầu, phần tử cuối chỉ đổi một phần (giá trị trung gian nguyên) để hiệu về đúng 0 — làm được vì mọi đóng góp là số nguyên và có thể chọn mọi mức `0..g_e`.

⇒ Đáp án = `|E|` nhỏ nhất có `Σ g ≥ d`. Với kích thước `k` cố định, tổng lớn nhất đạt khi chọn `k` đóng góp lớn nhất ⇒ `k*` = số phần tử đầu tiên của dãy giảm dần có tổng dồn `≥ d`. Cách 3 chỉ là cài đặt đếm của cùng phép cộng dồn này. Nếu tổng mọi `g < d` ⇒ `−1`. ∎

## 11. Phân tích Time / Space Complexity

Gọi `N = n1 + n2 ≤ 2·10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn phép gán | `O(6^N · N)` | `O(N)` |
| 1'. Thử từng `k` | `O(N² log N)` | `O(N)` |
| 2. Sắp xếp + cộng dồn | `O(N log N)` | `O(N)` |
| 3. Đếm | `O(N)` | `O(1)` |
| 4. Hai con trỏ | `O(N log N)` | `O(N)` |

In [14]:
rng = random.Random(1775)
A = [rng.randint(1, 3) for _ in range(10 ** 5)]; B = [rng.randint(4, 6) for _ in range(10 ** 5)]
for name, f in [("sort", ops_sort), ("count", Solution().minOperations), ("two_ptr", ops_two_pointers)]:
    t = time.perf_counter(); r = f(A, B); print(f"{name:<8} -> {r} ({(time.perf_counter()-t)*1000:.0f} ms)")

sort     -> 60012 (16 ms)
count    -> 60012 (6 ms)
two_ptr  -> 60012 (16 ms)


## 12. Bộ test

In [15]:
ALL = {"try_k": ops_try_k, "sort": ops_sort, "count": ops_count, "two_ptr": ops_two_pointers, "submit": Solution().minOperations}
def run_all(a, b, with_brute=False):
    res = {k: f(list(a), list(b)) for k, f in ALL.items()}
    if with_brute:
        res["brute"] = ops_bruteforce(a, b)
    assert len(set(res.values())) == 1, (a, b, res)
    return next(iter(res.values()))

for a, b, exp in examples:
    assert run_all(a, b) == exp
edge = [([1], [1], 0), ([1], [6], 1), ([6], [1], 1), ([1, 1], [6, 6], 2), ([1] * 7, [6], -1), ([6], [1] * 6, 0),
        ([6], [1] * 7, -1), ([3, 3], [2, 2, 2], 0)]
for a, b, exp in edge:
    assert run_all(a, b, with_brute=len(a) + len(b) <= 5) == exp, (a, b)
rng = random.Random(17750)
for _ in range(250):                                        # đối chiếu vét cạn thật
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 3))]; b = [rng.randint(1, 6) for _ in range(rng.randint(1, 5 - len(a) if len(a) < 5 else 1))]
    run_all(a, b, with_brute=True)
for _ in range(500):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 40))]; b = [rng.randint(1, 6) for _ in range(rng.randint(1, 40))]
    run_all(a, b)
print("Ví dụ, edge case, 250 test vét cạn, 500 test ngẫu nhiên: OK")

Ví dụ, edge case, 250 test vét cạn, 500 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ đổi phần tử ở **một** mảng

In [16]:
def wrong_one_array(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2: nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    gains = sorted([6 - x for x in nums1], reverse=True)       # SAI: bỏ qua mảng lớn
    for k, g in enumerate(gains, 1):
        d -= g
        if d <= 0: return k
    return 0 if d == 0 else -1

for a, b, exp in examples:
    print("sai :", wrong_one_array(a, b), "| đúng:", exp)

sai : 3 | đúng: 3
sai : -1 | đúng: -1
sai : -1 | đúng: 3


**Vì sao sai:** giảm phần tử của mảng tổng lớn cũng thu hẹp hiệu. **Sửa:** gộp đóng góp của **cả hai** mảng.

### Lỗi 2 — Quên đổi vai khi `sum(nums1) > sum(nums2)`

In [17]:
def wrong_no_swap(nums1, nums2):
    d = sum(nums2) - sum(nums1)                                  # có thể âm
    gains = sorted([6 - x for x in nums1] + [y - 1 for y in nums2], reverse=True)
    for k, g in enumerate(gains, 1):
        if d <= 0: return k - 1
        d -= g
    return -1 if d > 0 else len(gains)

a, b, exp = examples[2]
print("sai :", wrong_no_swap(a, b), "| đúng:", exp)

sai : 0 | đúng: 3


**Vì sao sai:** khi `nums1` có tổng lớn hơn, "đóng góp" phải tính ngược (`x − 1` cho `nums1`, `6 − y` cho `nums2`). **Sửa:** đổi vai để mảng thứ nhất luôn có tổng nhỏ hơn.

### Lỗi 3 — Đòi tổng đóng góp **đúng bằng** `d`

In [18]:
def wrong_exact(nums1, nums2):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2: nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    if d == 0: return 0
    gains = sorted([6 - x for x in nums1] + [y - 1 for y in nums2], reverse=True)
    acc = 0
    for k, g in enumerate(gains, 1):
        acc += g
        if acc == d: return k                                    # SAI: '>=' mới đúng
    return -1

a, b, exp = examples[0]
print("sai :", wrong_exact(a, b), "| đúng:", exp)

sai : -1 | đúng: 3


**Vì sao sai:** phần tử cuối có thể đổi **một phần** (giá trị trung gian), nên chỉ cần tổng `≥ d`. **Sửa:** `if acc >= d`.

### Lỗi 4 — Đếm thao tác theo `d // 5` (giả định mỗi thao tác luôn giảm 5)

In [19]:
def wrong_div5(nums1, nums2):
    d = abs(sum(nums1) - sum(nums2))
    return (d + 4) // 5

a, b = [3, 3, 3], [4, 4, 4, 4]             # ví dụ tự dựng (đối chiếu lại đề gốc); mọi đóng góp chỉ là 3
print("sai :", wrong_div5(a, b), "| đúng:", Solution().minOperations(a, b), "| thử từng k:", ops_try_k(a, b))
a, b = [1] * 7, [6]
print("Ví dụ 2 — sai :", wrong_div5(a, b), "| đúng:", Solution().minOperations(a, b))

sai : 2 | đúng: 3 | thử từng k: 3
Ví dụ 2 — sai : 1 | đúng: -1


**Vì sao sai:** đóng góp tối đa của mỗi phần tử phụ thuộc giá trị hiện tại (`3 → 6` chỉ +3, `4 → 1` chỉ −3), và có thể **không có** phần tử nào đóng góp 5: hiệu `d = 7` cần 3 thao tác chứ không phải 2. Còn bỏ qua trường hợp `−1`. **Sửa:** dùng đúng các đóng góp thực tế.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Làm hai tổng bằng nhau với ít thao tác nhất" ⇒ thu về **một hiệu `d`** và các **mức thu hẹp** của từng phần tử.
- Chọn ít phần tử nhất để tổng `≥ d` ⇒ **tham lam lấy lớn nhất trước**.
- Miền giá trị nhỏ ⇒ **đếm** thay cho sắp.
- Luôn kiểm tra khả thi bằng khoảng `[n, 6n]` trước.

## 15. Biến thể của bài

1. **Giá trị trong `[1, K]`**: thay 6 bằng `K`; đếm có `K` xô.
2. **Mỗi thao tác chỉ thay đổi ±1**: số thao tác = `d` (nếu khả thi) — bài khác hẳn.
3. **1658 Minimum Operations to Reduce X to Zero**, **2033 Minimum Operations to Make a Uni-Value Grid**: biến đổi tổng với thao tác có cấu trúc.
4. **Chi phí thao tác khác nhau theo mảng**: tham lam theo tỉ lệ đóng góp/chi phí (khi được đổi một phần thì giống fractional knapsack).
5. **Trả về chính các thao tác** (Bài tập 4).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Mọi phép gán | `O(6^N)` | `O(N)` | Kiểm chứng |
| 1'. Thử từng `k` | Tổng `k` lớn nhất | `O(N² log N)` | `O(N)` | Trung gian |
| 2. Sắp + cộng dồn | Tham lam | `O(N log N)` | `O(N)` | Dễ viết |
| 3. Đếm | Xô `0..5` | `O(N)` | `O(1)` | **Bản nộp** |
| 4. Hai con trỏ | Trộn hai dãy đóng góp | `O(N log N)` | `O(N)` | Khi muốn giữ hai mảng riêng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `([1,6], [2,5])`; (b) `([1], [2])`; (c) `([1,1,1], [6,6,6])`; (d) `([5,6,4,3,1,2], [6,3,3,1,4,5,3,4,1,3,4])`.

**Bài 2 (tính tay):** Với `nums1 = [2, 2]`, `nums2 = [5]`: tính `d`, danh sách đóng góp, và đáp án.

**Bài 3 (điền TODO):** Hoàn thiện `gain_counts(small, big)` trả mảng `cnt` 6 phần tử.

**Bài 4 (tự viết lại):** Viết `plan(nums1, nums2)` trả về **danh sách thao tác** `(tên_mảng, chỉ_số, giá_trị_mới)` với số thao tác tối thiểu; áp dụng chúng và kiểm tra hai tổng bằng nhau.

**Bài 5 (mini challenge):** Giá trị trong `[1, K]` (tham số `K`). Viết `min_ops_k(nums1, nums2, K)` `O(N + K)`, đối chiếu vét cạn nhỏ.

In [20]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def gain_counts_todo(small, big):
    cnt = [0] * 6
    # TODO: cnt[6 - x] += 1 cho x thuộc small; cnt[y - 1] += 1 cho y thuộc big
    return cnt

In [21]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def plan(nums1, nums2):
    # TODO: liệt kê (đóng góp, mảng, chỉ số), sắp giảm, đổi dần; phần tử cuối chỉ đổi đủ phần còn thiếu
    pass

def min_ops_k(nums1, nums2, K):
    # TODO: như Cách 3 với đóng góp K - x và y - 1, xô 0..K-1
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: so sánh tổng. Mức 2: đóng góp. Mức 3: (a) `0`; (b) `1`; (c) `3`; (d) `4`.

**Bài 2.** Mức 1: `s1 = 4 < s2 = 5`, `d = 1`. Mức 2: đóng góp `[4, 4]` (từ `nums1`) và `[4]` (từ `nums2`). Mức 3: một thao tác (`nums1[0]: 2 → 3`) ⇒ `1`.

**Bài 3.** Mức 1: hai vòng lặp. Mức 2: chỉ số xô là đóng góp. Mức 3: trả `cnt`.

**Bài 4.** Mức 1: đổi vai nhớ tên mảng gốc. Mức 2: với phần tử thuộc mảng nhỏ giá trị `x`, đóng góp `g`: giá trị mới `x + min(g, d)`; mảng lớn: `y − min(g, d)`. Mức 3: dừng khi `d = 0`; số thao tác phải bằng `Solution().minOperations`.

**Bài 5.** Mức 1: khả thi ⇔ `[n1, K·n1] ∩ [n2, K·n2] ≠ ∅`. Mức 2: xô `0..K−1`. Mức 3: duyệt `v` giảm dần với `⌈d/v⌉`.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2, 3
b1 = [Solution().minOperations(a, b) for a, b in [([1, 6], [2, 5]), ([1], [2]), ([1, 1, 1], [6, 6, 6]),
                                                   ([5, 6, 4, 3, 1, 2], [6, 3, 3, 1, 4, 5, 3, 4, 1, 3, 4])]]
print("Bài 1:", b1); assert b1 == [0, 1, 3, 4]
print("Bài 2:"); assert trace_count([2, 2], [5]) == 1
def gain_counts_todo(small, big):
    cnt = [0] * 6
    for x in small: cnt[6 - x] += 1
    for y in big: cnt[y - 1] += 1
    return cnt
print("Bài 3:", gain_counts_todo([1, 1, 2, 2, 2, 2], [1, 2, 3, 4, 5, 6])); assert gain_counts_todo([1, 1, 2, 2, 2, 2], [1, 2, 3, 4, 5, 6]) == [1, 1, 1, 1, 5, 3]

Bài 1: [0, 1, 3, 4]
Bài 2:
mảng tổng nhỏ=[2, 2] (tổng 4), mảng tổng lớn=[5] (tổng 5), d=1
cnt[v] (số phần tử có đóng góp v): [0, 0, 0, 0, 3, 0]
v=4: cần 1 <= có 3 -> xong, ops = 1
Bài 3: [1, 1, 1, 1, 5, 3]


In [23]:
# Bài 4, 5
def plan(nums1, nums2):
    arrs = {"nums1": list(nums1), "nums2": list(nums2)}
    small, big = ("nums1", "nums2") if sum(nums1) <= sum(nums2) else ("nums2", "nums1")
    d = sum(arrs[big]) - sum(arrs[small])
    items = [(6 - x, small, i) for i, x in enumerate(arrs[small])] + [(y - 1, big, i) for i, y in enumerate(arrs[big])]
    items.sort(key=lambda t: -t[0])
    ops = []
    for g, name, i in items:
        if d == 0: break
        use = min(g, d)
        if use == 0: break
        new = arrs[name][i] + use if name == small else arrs[name][i] - use
        ops.append((name, i, new)); d -= use
    return ops if d == 0 else None

def min_ops_k(nums1, nums2, K):
    s1, s2 = sum(nums1), sum(nums2)
    if s1 > s2: nums1, nums2, s1, s2 = nums2, nums1, s2, s1
    d = s2 - s1
    if d == 0: return 0
    cnt = [0] * K
    for x in nums1: cnt[K - x] += 1
    for y in nums2: cnt[y - 1] += 1
    ops = 0
    for v in range(K - 1, 0, -1):
        if cnt[v] == 0: continue
        need = (d + v - 1) // v
        if need <= cnt[v]: return ops + need
        ops += cnt[v]; d -= cnt[v] * v
    return -1

def brute_k(nums1, nums2, K):
    n1 = len(nums1); best = None
    for vals in product(range(1, K + 1), repeat=n1 + len(nums2)):
        if sum(vals[:n1]) == sum(vals[n1:]):
            c = sum(x != y for x, y in zip(vals, list(nums1) + list(nums2)))
            best = c if best is None else min(best, c)
    return -1 if best is None else best

rng = random.Random(6)
for _ in range(300):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 20))]; b = [rng.randint(1, 6) for _ in range(rng.randint(1, 20))]
    p = plan(a, b); ans = Solution().minOperations(a, b)
    if ans == -1:
        assert p is None
    else:
        arrs = {"nums1": list(a), "nums2": list(b)}
        for name, i, v in p:
            assert 1 <= v <= 6; arrs[name][i] = v
        assert len(p) == ans and sum(arrs["nums1"]) == sum(arrs["nums2"])
for _ in range(150):
    K = rng.randint(2, 5)
    a = [rng.randint(1, K) for _ in range(rng.randint(1, 2))]; b = [rng.randint(1, K) for _ in range(rng.randint(1, 3))]
    assert min_ops_k(a, b, K) == brute_k(a, b, K), (a, b, K)
print("Bài 4, 5: OK |", plan(*examples[0][:2]))

Bài 4, 5: OK | [('nums2', 0, 6), ('nums2', 1, 6), ('nums1', 5, 5)]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "thu hẹp hiệu bằng đóng góp lớn nhất":**

```python
if s1 > s2: swap
d = s2 - s1
cnt[6 - x] += 1 (mảng nhỏ); cnt[y - 1] += 1 (mảng lớn)
for v in 5..1:
    need = ceil(d / v)
    if need <= cnt[v]: return ops + need
    ops += cnt[v]; d -= cnt[v] * v
return -1
```

**Checklist:**
- [ ] Đổi vai để mảng thứ nhất có tổng **nhỏ hơn**?
- [ ] Gộp đóng góp của **cả hai** mảng?
- [ ] Dừng khi tổng đóng góp **≥ d** (không cần bằng)?
- [ ] Trả `0` khi hai tổng đã bằng nhau, `−1` khi hết đóng góp?
- [ ] Đã test hai mảng độ dài chênh lệch lớn (`1` vs `7`), tổng bằng nhau sẵn, `N = 2·10⁵`?